In [1]:
import numpy as np
from pythreejs import *
from ipywidgets import Output, VBox
from IPython.display import display

# Output area for interaction log
out = Output()
display(out)

# Generate random 3D vertices
vertices = np.random.rand(10, 3)

# Create small spheres for each vertex
spheres = []
for i, pos in enumerate(vertices):
    sphere = Mesh(
        geometry=SphereGeometry(radius=0.05),
        material=MeshLambertMaterial(color='red'),
        position=tuple(pos),
        name=str(i)
    )
    spheres.append(sphere)

# Setup scene
camera = PerspectiveCamera(position=[2, 2, 2], fov=60)
light = AmbientLight(intensity=0.5)
scene = Scene(children=[light] + spheres)

# Picker setup
picker = Picker(controlling=scene, event='click')
scene.children += (picker,)

# Handle click events
def on_pick(change):
    picked = change['new']
    if picked is not None and picked.object is not None:
        with out:
            print(f"Clicked on vertex {picked.object.name} at {picked.object.position}")

picker.observe(on_pick, names=['point'])

# Create renderer (include picker in controls!)
renderer = Renderer(
    camera=camera,
    scene=scene,
    controls=[OrbitControls(controlling=camera), picker],
    width=600,
    height=400
)

display(VBox([renderer, out]))


Output()

In [12]:
import open3d as o3d
import numpy as np

def create_sample_mesh():
    # Create a sample mesh (sphere here, but you can replace with your own)
    mesh = o3d.geometry.TriangleMesh.create_sphere(radius=1.0)
    mesh.compute_vertex_normals()
    return mesh

def create_vertex_markers(mesh, size=0.1):
    # Create larger spheres at each vertex to act as clickable markers
    markers = []
    for vertex in np.asarray(mesh.vertices):
        marker = o3d.geometry.TriangleMesh.create_sphere(radius=size)
        marker.translate(vertex)
        marker.paint_uniform_color([1, 0, 0])  # Red color for markers
        markers.append(marker)
    return markers

def pick_vertices(mesh):
    print("Instructions:")
    print("  - Press 'Shift + Left Click' to select vertices")
    print("  - Press 'Q' or close the window to finish selection")
    
    # Create larger spheres at the mesh vertices to make them easier to pick
    vertex_markers = create_vertex_markers(mesh)
    
    # Visualizer
    vis = o3d.visualization.VisualizerWithEditing()
    vis.create_window()
    vis.add_geometry(mesh)
    
    # Add vertex markers (larger red spheres) to the visualizer
    for marker in vertex_markers:
        vis.add_geometry(marker)
    
    # Run visualization and wait for user interaction
    vis.run()
    vis.destroy_window()

    # Retrieve picked points from the visualizer
    picked_points = vis.get_picked_points()
    
    print(f"\nPicked vertex indices: {picked_points}")
    
    return picked_points

# Run the workflow
mesh = create_sample_mesh()
picked_vertex_indices = pick_vertices(mesh)

# Display picked vertex positions
vertices = np.asarray(mesh.vertices)
print("\nPicked vertices:")
for idx in picked_vertex_indices:
    print(f"Vertex {idx}: {vertices[idx]}")


Instructions:
  - Press 'Shift + Left Click' to select vertices
  - Press 'Q' or close the window to finish selection

Picked vertex indices: []

Picked vertices:
